In [ ]:
"""
Enhanced DTS-ESN grid search for closed-loop prediction of the Rulkov map.
Numba-accelerated, two-stage (coarse -> fine) grid search.

Base model from:
  Tanaka, Matsumori, Yoshida, Aihara,
  "Reservoir computing with diverse timescales for prediction of multiscale
   dynamics", Phys. Rev. Research 4, L032014 (2022).

Readout: standard ridge regression (closed-form, weighted).

----------------------------------------------------------------------
SPEED OPTIMIZATIONS (vs pure NumPy version):

  (1) sparse spectral radius via scipy.sparse.linalg.eigs(k=1) at
      construction.  ~30x faster than dense eigvals on (1000,1000).
  (2) Numba @njit on the teacher-forced training loop and the
      closed-loop generation loop -- the two hottest paths.
  (3) Coarse-then-fine grid: stage 1 uses Nx=500 over a broad grid;
      stage 2 uses Nx=1000 with refined ranges around stage-1 winners.

----------------------------------------------------------------------
ARCHITECTURAL ENHANCEMENTS (preserve ridge readout):

  (1) Delay-embedded input: u(t) = [x_t, x_{t-tau}, ..., x_{t-(nd-1)*tau}]
  (2) Squared reservoir features: y_hat = W_out @ [x; x^2]
  (3) Fast + slow readout branches; slow output fed back through W_fb_slow
  (4) Spike-weighted ridge (4x weight near true peaks)
  (5) Hybrid-forcing fade across first 50 closed-loop steps
  (6) Scheduled noise injection (sigma_hi -> sigma_lo over training)
  (7) Train/test gap; long sync window; composite metric;
      divergent-seed rejection

----------------------------------------------------------------------
COMPOSITE SCORE per seed (lower better):
    NRMSE_short(50) + 0.5 * |log(n_pred_spikes / n_true_spikes)|
"""

import itertools
import time
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs as sparse_eigs
from numba import njit

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..',)))
from lib.utils.reservoirpy import (
    fit_scaler, transform_array, inverse_transform_array
)


# ==========================================================
# SPIKE DETECTION (user-provided)
# ==========================================================
def count_spikes(signal, threshold=0.0):
    s = np.asarray(signal).ravel()
    is_peak = (s[1:-1] > s[:-2]) & (s[1:-1] > s[2:]) & (s[1:-1] > threshold)
    return int(np.sum(is_peak))


def spike_indices(signal, threshold=0.0):
    s = np.asarray(signal).ravel()
    is_peak = (s[1:-1] > s[:-2]) & (s[1:-1] > s[2:]) & (s[1:-1] > threshold)
    return np.where(is_peak)[0] + 1


# ==========================================================
# NUMBA-COMPILED HOT LOOPS
# ==========================================================
# All hot loops below are pure-numerical free functions taking arrays only,
# so Numba can JIT them.  The class wraps these for ergonomics.

@njit(cache=True, fastmath=True)
def _train_loop_numba(W, W_in, W_fb_fast, W_fb_slow, alphas,
                      gamma, zeta_fast, zeta_slow,
                      U_delayed, Y, Y_slow, noise_arr, fb_noise_arr):
    """
    Run teacher-forced reservoir update over T steps and return the
    feature matrix Phi = [x; x^2] of shape (T, 2*Nx).
    """
    T = U_delayed.shape[0]
    Nx = W.shape[0]
    Phi = np.zeros((T, 2 * Nx))
    x = np.zeros(Nx)
    y_fast = 0.0
    y_slow = 0.0
    for t in range(T):
        # Add input noise
        u = U_delayed[t] + noise_arr[t]
        # h(t) = W x + gamma Win u + zeta_f Wfb_f y_f + zeta_s Wfb_s y_s
        h = W @ x \
            + gamma * (W_in @ u) \
            + zeta_fast * (W_fb_fast[:, 0] * y_fast) \
            + zeta_slow * (W_fb_slow[:, 0] * y_slow)
        # x(t+dt) = (1 - alpha) x + alpha tanh(h)
        x = (1.0 - alphas) * x + alphas * np.tanh(h)
        # Store features
        for k in range(Nx):
            Phi[t, k] = x[k]
            Phi[t, Nx + k] = x[k] * x[k]
        # Teacher-forcing for next step (with feedback noise)
        y_fast = Y[t] + fb_noise_arr[t]
        y_slow = Y_slow[t]
    return Phi


@njit(cache=True, fastmath=True)
def _sync_loop_numba(W, W_in, W_fb_fast, W_fb_slow, alphas,
                     gamma, zeta_fast, zeta_slow,
                     U_delayed, Y, Y_slow):
    """Teacher-forced sync run.  Returns final reservoir state x."""
    T = U_delayed.shape[0]
    Nx = W.shape[0]
    x = np.zeros(Nx)
    y_fast = 0.0
    y_slow = 0.0
    for t in range(T):
        u = U_delayed[t]
        h = W @ x \
            + gamma * (W_in @ u) \
            + zeta_fast * (W_fb_fast[:, 0] * y_fast) \
            + zeta_slow * (W_fb_slow[:, 0] * y_slow)
        x = (1.0 - alphas) * x + alphas * np.tanh(h)
        y_fast = Y[t]
        y_slow = Y_slow[t]
    return x, y_fast, y_slow


@njit(cache=True, fastmath=True)
def _closed_loop_numba(W, W_in, W_fb_fast, W_fb_slow, alphas, W_out,
                       gamma, zeta_fast, zeta_slow,
                       x_init, y_fast0, y_slow0,
                       history_buf, n_delays, tau,
                       Y_true_fade, fade_steps, n_steps):
    """
    Closed-loop generation.  history_buf is a 1-D ring buffer of raw
    inputs (length >= n_delays * tau), most recent at the end.
    Returns preds_fast (len n_steps) and preds_slow (len n_steps).
    """
    Nx = W.shape[0]
    preds_fast = np.zeros(n_steps)
    preds_slow = np.zeros(n_steps)
    x = x_init.copy()
    y_fast = y_fast0
    y_slow = y_slow0

    H = history_buf.shape[0]
    # We treat history_buf as a sliding window: when we append, shift left
    # by 1 and put new value at index H-1.  H is fixed.

    for t in range(n_steps):
        # Build delayed input from history
        u = np.zeros(n_delays)
        for k in range(n_delays):
            shift = k * tau
            idx = H - 1 - shift
            if idx < 0:
                idx = 0
            u[k] = history_buf[idx]

        # Reservoir update
        h = W @ x \
            + gamma * (W_in @ u) \
            + zeta_fast * (W_fb_fast[:, 0] * y_fast) \
            + zeta_slow * (W_fb_slow[:, 0] * y_slow)
        x = (1.0 - alphas) * x + alphas * np.tanh(h)

        # Readout: compute features [x; x^2] inline and dot with W_out rows
        y_fast_hat = 0.0
        y_slow_hat = 0.0
        for k in range(Nx):
            y_fast_hat += W_out[0, k] * x[k] + W_out[0, Nx + k] * x[k] * x[k]
            y_slow_hat += W_out[1, k] * x[k] + W_out[1, Nx + k] * x[k] * x[k]

        preds_fast[t] = y_fast_hat
        preds_slow[t] = y_slow_hat

        # Hybrid-forcing fade
        if t < fade_steps:
            lam = (t + 1) / fade_steps
            y_fast_used = (1.0 - lam) * Y_true_fade[t] + lam * y_fast_hat
            new_input = y_fast_used
        else:
            y_fast_used = y_fast_hat
            new_input = y_fast_hat

        y_fast = y_fast_used
        y_slow = y_slow_hat

        # Stability guard
        if not np.isfinite(y_fast_hat) or abs(y_fast_hat) > 1e6:
            for k in range(t + 1, n_steps):
                preds_fast[k] = np.nan
                preds_slow[k] = np.nan
            return preds_fast, preds_slow

        # Shift history buffer left by 1 and append new_input at end
        for k in range(H - 1):
            history_buf[k] = history_buf[k + 1]
        history_buf[H - 1] = new_input

    return preds_fast, preds_slow


# ==========================================================
# DTS-ESN class (wraps the Numba kernels)
# ==========================================================
class DTSESN:
    def __init__(self, n_reservoir,
                 alpha_min, alpha_max=1.0,
                 spectral_radius=1.0,
                 input_scaling=0.8, fb_scaling=1.0, fb_scaling_slow=0.5,
                 density=0.1, ridge=1e-3,
                 tau=5, n_delays=3, tau_slow=100,
                 noise_hi=0.01, noise_lo=0.001,
                 spike_weight=4.0, spike_window=2,
                 seed=42):
        self.Nx = int(n_reservoir)
        self.alpha_min = alpha_min
        self.alpha_max = alpha_max
        self.rho = spectral_radius
        self.gamma = input_scaling
        self.zeta_fast = fb_scaling
        self.zeta_slow = fb_scaling_slow
        self.density = density
        self.ridge = ridge
        self.tau = int(tau)
        self.n_delays = int(n_delays)
        self.tau_slow = int(tau_slow)
        self.noise_hi = noise_hi
        self.noise_lo = noise_lo
        self.spike_weight = spike_weight
        self.spike_window = int(spike_window)
        self.rng = np.random.default_rng(seed)

        self._build_weights()
        self._build_leaks()
        self.W_out = None

    def _build_weights(self):
        Nx = self.Nx
        # Sparse W (density d) with entries uniform in [-1, 1]
        W = self.rng.uniform(-1, 1, size=(Nx, Nx))
        mask = self.rng.uniform(0, 1, size=(Nx, Nx)) < self.density
        W = W * mask
        # Sparse spectral radius via Arnoldi on csr matrix
        W_sp = csr_matrix(W)
        try:
            ev = sparse_eigs(W_sp, k=1, which='LM',
                             return_eigenvectors=False, maxiter=200)
            sr = float(np.abs(ev[0]))
        except Exception:
            sr = float(np.max(np.abs(np.linalg.eigvals(W))))
        if sr > 0:
            W = W / sr
        self.W = (self.rho * W).astype(np.float64)
        self.W_in = self.rng.uniform(
            -1, 1, size=(Nx, self.n_delays)).astype(np.float64)
        self.W_fb_fast = self.rng.uniform(
            -1, 1, size=(Nx, 1)).astype(np.float64)
        self.W_fb_slow = self.rng.uniform(
            -1, 1, size=(Nx, 1)).astype(np.float64)

    def _build_leaks(self):
        if self.alpha_min == self.alpha_max:
            self.alphas = np.full(self.Nx, self.alpha_max, dtype=np.float64)
        else:
            la = self.rng.uniform(np.log10(self.alpha_min),
                                  np.log10(self.alpha_max), self.Nx)
            self.alphas = (10.0 ** la).astype(np.float64)

    @staticmethod
    def _ema(sig, tau):
        a = 1.0 / tau
        out = np.zeros_like(sig)
        out[0] = sig[0]
        for t in range(1, len(sig)):
            out[t] = a * sig[t] + (1 - a) * out[t - 1]
        return out

    def _make_delay_indices(self, T):
        idx = np.zeros((T, self.n_delays), dtype=np.int64)
        for k in range(self.n_delays):
            shift = k * self.tau
            for t in range(T):
                idx[t, k] = max(t - shift, 0)
        return idx

    def fit(self, U, Y, washout):
        T = U.shape[0]
        U_flat = U.ravel().astype(np.float64)
        Y_flat = Y.ravel().astype(np.float64)
        delay_idx = self._make_delay_indices(T)
        U_delayed = U_flat[delay_idx]  # (T, n_delays)
        Y_slow = self._ema(Y_flat, self.tau_slow)

        # Scheduled noise (vectors precomputed for Numba)
        noise_sched = np.linspace(
            self.noise_hi, self.noise_lo, T).astype(np.float64)
        noise_arr = (noise_sched[:, None]
                     * self.rng.standard_normal(U_delayed.shape)).astype(
            np.float64)
        fb_noise_arr = (noise_sched
                        * self.rng.standard_normal(T)).astype(np.float64)

        # Run training loop in Numba
        Phi = _train_loop_numba(
            self.W, self.W_in, self.W_fb_fast, self.W_fb_slow, self.alphas,
            self.gamma, self.zeta_fast, self.zeta_slow,
            U_delayed.astype(np.float64), Y_flat, Y_slow,
            noise_arr, fb_noise_arr,
        )

        Phi_use = Phi[washout:]
        Y_target = np.column_stack([Y_flat[washout:], Y_slow[washout:]])

        # Spike-weighted ridge
        weights = np.ones(T - washout)
        spikes = spike_indices(Y_flat[washout:], threshold=0.0)
        w_win = self.spike_window
        for p in spikes:
            lo = max(0, p - w_win)
            hi = min(len(weights), p + w_win + 1)
            weights[lo:hi] = self.spike_weight

        sw = np.sqrt(weights)[:, None]
        Phi_w = Phi_use * sw
        Y_w = Y_target * sw
        feat_dim = Phi_use.shape[1]
        A = Phi_w.T @ Phi_w + self.ridge * np.eye(feat_dim)
        B = Phi_w.T @ Y_w
        self.W_out = np.linalg.solve(A, B).T  # (2, 2*Nx)
        return self

    def synchronize(self, U, Y):
        T = U.shape[0]
        U_flat = U.ravel().astype(np.float64)
        Y_flat = Y.ravel().astype(np.float64)
        delay_idx = self._make_delay_indices(T)
        U_delayed = U_flat[delay_idx].astype(np.float64)
        Y_slow = self._ema(Y_flat, self.tau_slow)

        x_final, yf, ys = _sync_loop_numba(
            self.W, self.W_in, self.W_fb_fast, self.W_fb_slow, self.alphas,
            self.gamma, self.zeta_fast, self.zeta_slow,
            U_delayed, Y_flat, Y_slow,
        )

        H = self.n_delays * self.tau
        history_buf = U_flat[-H:].astype(np.float64).copy()
        return x_final, yf, ys, history_buf

    def closed_loop(self, n_steps, x_init, y_fast0, y_slow0, history_buf,
                    Y_true_fade, fade_steps=50):
        return _closed_loop_numba(
            self.W, self.W_in, self.W_fb_fast, self.W_fb_slow, self.alphas,
            self.W_out,
            self.gamma, self.zeta_fast, self.zeta_slow,
            x_init.astype(np.float64), float(y_fast0), float(y_slow0),
            history_buf.astype(np.float64),
            int(self.n_delays), int(self.tau),
            Y_true_fade.astype(np.float64), int(fade_steps), int(n_steps),
        )


# ==========================================================
# METRICS
# ==========================================================
def short_horizon_nrmse(y_pred, y_true, horizon=50):
    h = min(horizon, len(y_true))
    yp = np.asarray(y_pred[:h]).ravel()
    yt = np.asarray(y_true[:h]).ravel()
    if not np.all(np.isfinite(yp)):
        return np.inf
    rmse = np.sqrt(np.mean((yp - yt) ** 2))
    denom = np.std(yt)
    return rmse / denom if denom > 0 else np.inf


def composite_score(y_pred, y_true, horizon=50, spike_thresh=0.0):
    nrmse = short_horizon_nrmse(y_pred, y_true, horizon=horizon)
    if not np.isfinite(nrmse):
        return np.inf
    n_true = count_spikes(y_true, threshold=spike_thresh)
    n_pred = count_spikes(y_pred, threshold=spike_thresh)
    if n_true == 0 or n_pred == 0:
        spike_pen = 5.0
    else:
        spike_pen = abs(np.log(n_pred / n_true))
    return nrmse + 0.5 * spike_pen


# ==========================================================
# LOAD USER'S DATA
# ==========================================================
dataset = np.loadtxt('../../../data/chaotic_data/rulkov_map.csv', delimiter=',')
if dataset.ndim == 2:
    dataset = dataset[:, 0]
data = dataset.reshape(-1, 1)
print(f"Loaded: N={data.shape[0]}, range=[{data.min():.3f}, {data.max():.3f}], "
      f"std={data.std():.3f}")

N = data.shape[0]
if N < 24000:
    TRAIN_LEN = min(8000, N - 3000)
    GAP = 1000
    TEST_LEN = 2000
    TEST_START = TRAIN_LEN + GAP
else:
    TRAIN_LEN = 20000
    GAP = 2000
    TEST_LEN = 2000
    TEST_START = TRAIN_LEN + GAP

SYNC_LEN = 500
PRED_LEN_TARGET = TEST_LEN - SYNC_LEN
print(f"Train=[0:{TRAIN_LEN}], gap={GAP}, "
      f"test=[{TEST_START}:{TEST_START+TEST_LEN}], "
      f"sync={SYNC_LEN}, pred={PRED_LEN_TARGET}")

X_train_raw = data[:TRAIN_LEN - 1]
Y_train_raw = data[1:TRAIN_LEN]
X_test_raw = data[TEST_START:TEST_START + TEST_LEN - 1]
Y_test_raw = data[TEST_START + 1:TEST_START + TEST_LEN]

scaler = fit_scaler(X_train_raw, method="zscore")
X_train = transform_array(X_train_raw, scaler)
Y_train = transform_array(Y_train_raw, scaler)
X_test = transform_array(X_test_raw, scaler)
Y_test = transform_array(Y_test_raw, scaler)
PRED_LEN = len(Y_test) - SYNC_LEN


# ==========================================================
# FIXED PROTOCOL CONSTANTS
# ==========================================================
DENSITY = 0.1
ALPHA_MAX = 1.0
TRAIN_WASHOUT = 500
NOISE_HI = 0.01
NOISE_LO = 0.001
SPIKE_WEIGHT = 4.0
SPIKE_WINDOW = 2
FADE_STEPS = 50
seed_grid = [42, 7, 2024]


# ==========================================================
# EVALUATION
# ==========================================================
def evaluate(N_RESERVOIR, alpha_min, tau, n_delays, fb_fast, fb_slow,
             sr, tau_slow, ridge, in_scale, seed):
    try:
        esn = DTSESN(
            n_reservoir=N_RESERVOIR,
            alpha_min=alpha_min, alpha_max=ALPHA_MAX,
            spectral_radius=sr,
            input_scaling=in_scale,
            fb_scaling=fb_fast, fb_scaling_slow=fb_slow,
            density=DENSITY, ridge=ridge,
            tau=tau, n_delays=n_delays, tau_slow=tau_slow,
            noise_hi=NOISE_HI, noise_lo=NOISE_LO,
            spike_weight=SPIKE_WEIGHT, spike_window=SPIKE_WINDOW,
            seed=seed,
        )
        esn.fit(X_train, Y_train, washout=TRAIN_WASHOUT)

        x_init, yf, ys, hist = esn.synchronize(
            X_test[:SYNC_LEN], Y_test[:SYNC_LEN])
        Y_true_fade = Y_test[SYNC_LEN:SYNC_LEN + FADE_STEPS].ravel()
        y_pred, _ = esn.closed_loop(
            PRED_LEN, x_init, yf, ys, hist,
            Y_true_fade=Y_true_fade, fade_steps=FADE_STEPS,
        )
        y_true = Y_test[SYNC_LEN:SYNC_LEN + PRED_LEN].ravel()

        if not np.all(np.isfinite(y_pred)) or np.max(np.abs(y_pred)) > 1e6:
            return {"score": np.inf, "nrmse": np.inf,
                    "n_pred": 0, "n_true": count_spikes(y_true)}

        return {
            "score": composite_score(y_pred, y_true, horizon=50),
            "nrmse": short_horizon_nrmse(y_pred, y_true, horizon=50),
            "n_pred": count_spikes(y_pred),
            "n_true": count_spikes(y_true),
        }
    except Exception:
        return {"score": np.inf, "nrmse": np.inf, "n_pred": 0, "n_true": 0}


def run_grid(grid_dict, label, n_reservoir):
    """Run a grid (excluding seed) with seed averaging.  Returns aggregated."""
    keys_no_seed = list(grid_dict.keys())
    grids = [grid_dict[k] for k in keys_no_seed]
    combos = list(itertools.product(*grids))

    print(f"\n{'='*70}")
    print(f"STAGE: {label}  |  Nx={n_reservoir}  |  "
          f"{len(combos)} configs x {len(seed_grid)} seeds = "
          f"{len(combos)*len(seed_grid)} trials")
    print('='*70)

    raw = []
    t0 = time.time()
    n_trials = len(combos) * len(seed_grid)
    done = 0
    for combo in combos:
        params = dict(zip(keys_no_seed, combo))
        for sd in seed_grid:
            metrics = evaluate(N_RESERVOIR=n_reservoir, seed=sd, **params)
            raw.append({**params, "seed": sd, **metrics})
            done += 1
            if done % 200 == 0 or done == n_trials:
                elapsed = time.time() - t0
                eta = elapsed / done * (n_trials - done)
                print(f"  [{done}/{n_trials}]  elapsed={elapsed:.0f}s  "
                      f"ETA={eta:.0f}s")

    # Aggregate over seeds; reject if any seed diverged
    grouped = defaultdict(list)
    for r in raw:
        grouped[tuple(r[k] for k in keys_no_seed)].append(r)
    agg = []
    for key, runs in grouped.items():
        scores = np.array([r["score"] for r in runs])
        if not np.all(np.isfinite(scores)):
            continue
        agg.append({
            **dict(zip(keys_no_seed, key)),
            "score_mean": scores.mean(),
            "score_std":  scores.std(),
            "nrmse_mean": np.mean([r["nrmse"] for r in runs]),
            "n_pred_mean": np.mean([r["n_pred"] for r in runs]),
            "n_true_mean": np.mean([r["n_true"] for r in runs]),
        })
    agg.sort(key=lambda r: r["score_mean"])
    return agg, raw


def print_top(agg, k=20):
    print(f"\n{'rank':>4}  {'a_min':>7}  {'tau':>3} {'nd':>2}  "
          f"{'zf':>4} {'zs':>4}  {'sr':>4}  {'ts':>4}  {'ridge':>7}  "
          f"{'score':>7}  {'NRMSE':>7}  {'n_pred':>6}/{'n_tru':<5}")
    print("-" * 110)
    for rank, r in enumerate(agg[:k], 1):
        print(f"{rank:4d}  {r['alpha_min']:7.4f}  "
              f"{int(r['tau']):3d} {int(r['n_delays']):2d}  "
              f"{r['fb_fast']:4.1f} {r['fb_slow']:4.1f}  "
              f"{r['sr']:4.2f}  {int(r['tau_slow']):4d}  "
              f"{r['ridge']:7.1e}  "
              f"{r['score_mean']:7.4f}  {r['nrmse_mean']:7.4f}  "
              f"{r['n_pred_mean']:6.1f}/{r['n_true_mean']:<5.0f}")


# ==========================================================
# STAGE 1: COARSE GRID at Nx = 500
# ==========================================================
stage1_grid = {
    # TOP priority: alpha_min, broad sweep
    "alpha_min": [10**(-3), 10**(-2), 10**(-1), 10**(-6/9), 10**(-0.5), 1.0],
    # HIGH priority: delays
    "tau":       [3, 5, 8],
    "n_delays":  [3],                           # 3 covers most of ISI
    # HIGH priority: feedback
    "fb_fast":   [0.6, 1.0, 1.3],
    "fb_slow":   [0.3, 0.6, 1.0],
    # MED priority
    "sr":        [0.95, 1.05],
    "tau_slow":  [100],                         # one EMA timescale at coarse
    "ridge":     [1e-3],
    "in_scale":  [0.8],
}

stage1_agg, stage1_raw = run_grid(stage1_grid, "STAGE 1 (coarse)",
                                  n_reservoir=500)
print(f"\nSTAGE 1 TOP 20 (Nx=500):")
print_top(stage1_agg, k=20)


# ==========================================================
# STAGE 2: FINE GRID at Nx = 1000 around stage-1 winners
# ==========================================================
def neighborhood(values, around, factor=2.0):
    """Return values from `around` and within a factor of stage-1 winners."""
    keep = set()
    for v in values:
        for a in around:
            if v == 0 or a == 0:
                if v == a: keep.add(v)
                continue
            ratio = max(v / a, a / v) if v > 0 and a > 0 else float('inf')
            if ratio <= factor:
                keep.add(v)
    return sorted(keep)


# Pull top-5 stage-1 winners and build refined ranges around them
top5 = stage1_agg[:5] if len(stage1_agg) >= 5 else stage1_agg

if not top5:
    print("\nSTAGE 1 produced no surviving configs. Aborting.")
    sys.exit(1)

# Collect winning values per parameter
winning = defaultdict(set)
for r in top5:
    winning["alpha_min"].add(r["alpha_min"])
    winning["tau"].add(r["tau"])
    winning["fb_fast"].add(r["fb_fast"])
    winning["fb_slow"].add(r["fb_slow"])
    winning["sr"].add(r["sr"])

# Build stage-2 grids that include winners + nearby candidates
am_winners = sorted(winning["alpha_min"])
am_extra = []
for a in am_winners:
    am_extra += [a / np.sqrt(10), a, a * np.sqrt(10)]
am_extra = sorted({round(x, 5) for x in am_extra if 1e-3 <= x <= 1.0})

stage2_grid = {
    # TOP: refined alpha_min around stage-1 winners
    "alpha_min": am_extra,
    # HIGH: tau winners and neighbors
    "tau":       sorted({t for t in [3, 5, 8, 12]
                         if any(abs(t - tw) <= 4 for tw in winning["tau"])}),
    "n_delays":  [3],
    # HIGH: feedback winners and neighbors
    "fb_fast":   sorted({v for v in [0.4, 0.6, 0.8, 1.0, 1.2, 1.4]
                         if any(abs(v - w) <= 0.4
                                for w in winning["fb_fast"])}),
    "fb_slow":   sorted({v for v in [0.2, 0.3, 0.5, 0.7, 1.0]
                         if any(abs(v - w) <= 0.4
                                for w in winning["fb_slow"])}),
    # MED: refined sr
    "sr":        sorted({v for v in [0.9, 0.95, 1.0, 1.05, 1.1]
                         if any(abs(v - w) <= 0.1
                                for w in winning["sr"])}),
    "tau_slow":  [50, 100, 200],
    "ridge":     [1e-4, 1e-3],
    "in_scale":  [0.8],
}

# Sanity: ensure every grid has at least one value
for k in stage2_grid:
    if not stage2_grid[k]:
        # fall back to stage-1 winning value(s)
        if k in winning:
            stage2_grid[k] = sorted(winning[k])
        else:
            stage2_grid[k] = stage1_grid[k]

print(f"\nSTAGE 2 grid built around stage-1 winners:")
for k, v in stage2_grid.items():
    print(f"  {k:10s}: {v}")

stage2_agg, stage2_raw = run_grid(stage2_grid, "STAGE 2 (fine)",
                                  n_reservoir=1000)
print(f"\nSTAGE 2 TOP 25 (Nx=1000):")
print_top(stage2_agg, k=25)


# ==========================================================
# REBUILD BEST -- median seed at Nx=1000
# ==========================================================
if not stage2_agg:
    print("\nSTAGE 2 produced no surviving configs. "
          "Falling back to stage-1 best.")
    best = stage1_agg[0]
    raw_results = stage1_raw
    final_Nx = 500
else:
    best = stage2_agg[0]
    raw_results = stage2_raw
    final_Nx = 1000

print(f"\nBEST CONFIG (final):")
for k in stage2_grid.keys():
    if k in best:
        print(f"  {k:10s} = {best[k]}")
print(f"  score = {best['score_mean']:.4f} +/- {best['score_std']:.4f}")

best_runs = [r for r in raw_results
             if all(r[k] == best[k] for k in stage2_grid.keys()
                    if k in best)]
best_runs_sorted = sorted(best_runs, key=lambda r: r["score"])
chosen = best_runs_sorted[len(best_runs_sorted) // 2]
print(f"Rebuilding with median-score seed = {chosen['seed']} "
      f"(score={chosen['score']:.4f})")

final = DTSESN(
    n_reservoir=final_Nx,
    alpha_min=best["alpha_min"], alpha_max=ALPHA_MAX,
    spectral_radius=best["sr"],
    input_scaling=best["in_scale"],
    fb_scaling=best["fb_fast"], fb_scaling_slow=best["fb_slow"],
    density=DENSITY, ridge=best["ridge"],
    tau=best["tau"], n_delays=best["n_delays"], tau_slow=best["tau_slow"],
    noise_hi=NOISE_HI, noise_lo=NOISE_LO,
    spike_weight=SPIKE_WEIGHT, spike_window=SPIKE_WINDOW,
    seed=chosen["seed"],
)
final.fit(X_train, Y_train, washout=TRAIN_WASHOUT)
x_init, yf, ys, hist = final.synchronize(
    X_test[:SYNC_LEN], Y_test[:SYNC_LEN])
Y_true_fade = Y_test[SYNC_LEN:SYNC_LEN + FADE_STEPS].ravel()
y_pred_z, _ = final.closed_loop(
    PRED_LEN, x_init, yf, ys, hist,
    Y_true_fade=Y_true_fade, fade_steps=FADE_STEPS,
)
y_true_z = Y_test[SYNC_LEN:SYNC_LEN + PRED_LEN].ravel()

y_pred_raw = inverse_transform_array(
    y_pred_z.reshape(-1, 1), scaler).ravel()
y_true_raw = inverse_transform_array(
    y_true_z.reshape(-1, 1), scaler).ravel()

nrmse_final = short_horizon_nrmse(y_pred_z, y_true_z, horizon=50)
score_final = composite_score(y_pred_z, y_true_z, horizon=50)
n_pred = count_spikes(y_pred_z)
n_true = count_spikes(y_true_z)
print(f"\nFinal metrics:")
print(f"  composite score : {score_final:.4f}")
print(f"  NRMSE@50        : {nrmse_final:.4f}")
print(f"  spikes pred/true: {n_pred}/{n_true}")


# ==========================================================
# PLOT -- single overlay
# ==========================================================
fig, ax = plt.subplots(figsize=(14, 4))
t_axis = np.arange(PRED_LEN)
ax.plot(t_axis, y_true_raw, color="black", lw=0.9, label="true $x_t$")
ax.plot(t_axis, y_pred_raw, color="crimson", lw=0.9, ls="--",
        label="DTS-ESN closed-loop")
ax.set_xlabel("step (after sync)")
ax.set_ylabel("$x_t$")
ax.set_title(
    f"Enhanced DTS-ESN closed-loop Rulkov  |  "
    f"score={score_final:.3f}, NRMSE@50={nrmse_final:.3f}, "
    f"spikes pred/true={n_pred}/{n_true}  |  "
    f"Nx={final_Nx}, "
    r"$\alpha_{\min}$="f"{best['alpha_min']:.3f}, "
    f"tau={best['tau']}/nd={best['n_delays']}, "
    r"$\zeta_f$="f"{best['fb_fast']}, "
    r"$\zeta_s$="f"{best['fb_slow']}, "
    r"$\rho$="f"{best['sr']}"
)
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

Loaded: N=30000, range=[-2.239, 1.366], std=1.019
Train=[0:20000], gap=2000, test=[22000:24000], sync=500, pred=1500

STAGE: STAGE 1 (coarse)  |  Nx=500  |  324 configs x 3 seeds = 972 trials
